# Building Heights: Global Building Atlas vs Cadastral

- Cadastral as primary source, GBA as fallback for missing heights and missing buildings.

**Inputs (from DuckDB `raw` schema):**
- `raw.{CITY}_buildings` — cadastral building polygons
- `raw.{CITY}_gba`       — Global Building Atlas polygons
- `raw.{CITY}_boundary`  — municipal boundary

**Output (to DuckDB `preprocessing` schema):**
- `preprocessing.{CITY}_buildings_cadastral` — cleaned cadastral layer
- `preprocessing.{CITY}_buildings_gba`       — cleaned GBA layer
- `preprocessing.{CITY}_buildings`           — merged final layer (cadastral + GBA gap-fill)

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import duckdb
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely.wkb
from shapely import wkt as shapely_wkt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from config import ROOT_PATH, DB_PATH, get_city_cfg, CRS_METRIC, CRS_WGS, gdf_to_duckdb, load_table, gpkg_to_duckdb

os.chdir(ROOT_PATH)


In [ ]:
CITY = "milano"
cfg  = get_city_cfg(CITY)
src  = cfg["data_sources"]

con = duckdb.connect(str(DB_PATH))
con.execute("LOAD spatial;")
print(f"Connected to {DB_PATH}")

## 1. Load Input Data
Load from file → save to `raw` schema → use loaded GDFs directly.
If a table already exists in `raw`, read from DuckDB instead of re-loading from file.

In [ ]:
def table_exists(con, schema: str, table: str) -> bool:
    result = con.execute(
        f"SELECT count(*) FROM information_schema.tables "
        f"WHERE table_schema='{schema}' AND table_name='{table}'"
    ).fetchone()[0]
    return result > 0


def load_or_ingest(con, path: str, schema: str, table: str, city: str) -> gpd.GeoDataFrame:
    """Return GeoDataFrame from DuckDB if already ingested, else load from file and save."""
    if table_exists(con, schema, table):
        print(f"  {schema}.{table}: reading from DuckDB")
        return load_table(con, schema, table)
    else:
        print(f"  {schema}.{table}: ingesting from {path}")
        return gpkg_to_duckdb(con, path, f"{schema}.{table}", city)


boundary        = load_or_ingest(con, src["boundary"],  "raw", f"{CITY}_boundary",  CITY)
cadastral_raw   = load_or_ingest(con, src["building_layers"]["cadastral"], "raw", f"{CITY}_cadastral", CITY)
global_atlas    = load_or_ingest(con, cfg["building_layers"]["gba"], "raw", f"{CITY}_gba", CITY)

print(f"\nBoundary:   {len(boundary):,} features")
print(f"Cadastral:  {len(cadastral_raw):,} features")
print(f"GBA:        {len(global_atlas):,} features")
print(f"Columns (cadastral): {cadastral_raw.columns.tolist()}")

## 2. Clean & Clip

- Clip both datasets to the municipal boundary
- Filter buildings with footprint ≤ 5 m² (removes shelters, garages, artefacts)
- Cadastral height = `altezza_gr`, fallback to `quota_gron - quota_pied` where missing

In [ ]:
# GBA
global_atlas = gpd.clip(global_atlas, boundary)
global_atlas["area_m2"] = global_atlas.geometry.area
global_atlas = global_atlas[global_atlas["area_m2"] > 5].copy()
global_atlas["height_filtered"] = global_atlas["height"].fillna(0)

# Cadastral
cadastral_clipped = gpd.clip(cadastral_raw, boundary)
cadastral_clipped["area_m2"] = cadastral_clipped.geometry.area
cadastral_clipped = cadastral_clipped[cadastral_clipped["area_m2"] > 5].copy()
cadastral_clipped["height"] = cadastral_clipped["altezza_gr"].fillna(
    cadastral_clipped["quota_gron"] - cadastral_clipped["quota_pied"]
)
cadastral_clipped["height_filtered"] = cadastral_clipped["height"].where(
    cadastral_clipped["height"] > 0, 0
)

print(f"GBA clipped:       {len(global_atlas):,}")
print(f"Cadastral clipped: {len(cadastral_clipped):,}")

## 3. Geometry Matching
Spatial join by intersection; keep the candidate with the largest overlap area per building.

In [ ]:
left  = global_atlas[["geometry", "height_filtered"]].rename(columns={"height_filtered": "height_left"})
right = cadastral_clipped[["geometry", "height_filtered"]].rename(columns={"height_filtered": "height_right"})

joined_raw = gpd.sjoin(left, right, how="inner", predicate="intersects")

right_geoms = right.geometry
joined_raw["overlap_area"] = joined_raw.apply(
    lambda row: row.geometry.intersection(right_geoms.loc[row["index_right"]]).area,
    axis=1,
)

# One row per GBA building (best overlap) — used for height comparison
best_per_gba = (
    joined_raw
    .sort_values("overlap_area", ascending=False)
    .loc[lambda df: ~df.index.duplicated(keep="first")]
)

# One row per cadastral building — used for coverage counts
best_per_cad = (
    joined_raw
    .sort_values("overlap_area", ascending=False)
    .loc[lambda df: ~df["index_right"].duplicated(keep="first")]
)

joined_all          = best_per_gba.copy()
n_gba_matched       = best_per_gba.index.nunique()
n_cadastral_matched = best_per_cad["index_right"].nunique()

print("GEOMETRY MATCHING")
print(f"  Total matched (best overlap per GBA) : {len(joined_all):>8,}")
print(f"  GBA  matched : {n_gba_matched:,}  |  only : {len(global_atlas) - n_gba_matched:,}")
print(f"  CAD  matched : {n_cadastral_matched:,}  |  only : {len(cadastral_clipped) - n_cadastral_matched:,}")

## 4. Height Comparison
Compare heights only where **both** datasets have height > 0.

In [ ]:
joined_heights = joined_all[
    (joined_all["height_left"]  > 0) &
    (joined_all["height_right"] > 0)
].copy()
joined_heights["height_diff"] = (joined_heights["height_left"] - joined_heights["height_right"]).abs()
joined_heights["signed_diff"] =  joined_heights["height_left"] - joined_heights["height_right"]

thresholds   = [1, 2, 3, 5, 8, 10]
mean_bias    = joined_heights["signed_diff"].mean()
median       = joined_heights["height_diff"].median()
mad          = np.median(np.abs(joined_heights["height_diff"] - median))
threshold_75 = joined_heights["height_diff"].quantile(0.75)

n_geom_gba_has_height  = (joined_all["height_left"]  > 0).sum()
n_geom_cad_has_height  = (joined_all["height_right"] > 0).sum()
n_height_in_both       = len(joined_heights)

print("HEIGHT COMPARISON")
print(f"  Geometrically matched             : {len(joined_all):>8,}")
print(f"  Matched + height in GBA           : {n_geom_gba_has_height:>8,}")
print(f"  Matched + height in cadastral     : {n_geom_cad_has_height:>8,}")
print(f"  Matched + height in BOTH          : {n_height_in_both:>8,}  ← used for comparison")
print()
for t in thresholds:
    n   = (joined_heights["height_diff"] <= t).sum()
    pct = (joined_heights["height_diff"] <= t).mean() * 100
    print(f"  +/-{t:>2} m  =>  {n:>6,}  ({pct:.1f}%)")
print(f"  Mean signed bias (GBA - cadastral): {mean_bias:>+8.2f} m")
print(f"  75th percentile of differences    : {threshold_75:>8.2f} m")
print(f"  Robust MAD threshold              : {median + 2*mad:>8.2f} m")
print(joined_heights["height_diff"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).round(2).to_string())

## 5. Coverage Summary

In [ ]:
# GBA buildings with no cadastral geometry match
matched_gba      = gpd.sjoin(global_atlas[["geometry", "height_filtered"]],
                             cadastral_clipped[["geometry"]], how="left", predicate="intersects")
gba_only_idx     = matched_gba[matched_gba["index_right"].isna()].index.unique()
gba_no_cadastral = global_atlas.loc[gba_only_idx].copy()

# Cadastral buildings with no GBA match
matched_cadastral   = gpd.sjoin(cadastral_clipped[["geometry"]],
                                global_atlas[["geometry"]], how="left", predicate="intersects")
cadastral_only_idx  = matched_cadastral[matched_cadastral["index_right"].isna()].index.unique()

n_gba_matched_cov       = matched_gba[matched_gba["index_right"].notna()].index.nunique()
n_cadastral_matched_cov = matched_cadastral[matched_cadastral["index_right"].notna()].index.nunique()
n_gba_only              = len(global_atlas)      - n_gba_matched_cov
n_cadastral_only        = len(cadastral_clipped) - n_cadastral_matched_cov

pct_gba_matched  = n_gba_matched_cov       / len(global_atlas)      * 100
pct_cad_matched  = n_cadastral_matched_cov / len(cadastral_clipped) * 100
pct_gba_only     = n_gba_only              / len(global_atlas)      * 100
pct_cad_only     = n_cadastral_only        / len(cadastral_clipped) * 100
categories       = ["GBA", "Cadastral"]

print("COVERAGE SUMMARY (area > 5 m², height > 0)")
print(f"  1. Height in cadastral (total)        : {(cadastral_clipped['height_filtered'] > 0).sum():>8,}  (of {len(cadastral_clipped):,})")
print(f"  2. Height in GBA (total)              : {(global_atlas['height_filtered'] > 0).sum():>8,}  (of {len(global_atlas):,})")
print(f"  3. Height in both (geom + height > 0) : {n_height_in_both:>8,}")
print(f"  4. GBA only (all polygons)            : {n_gba_only:>8,}")
print(f"  5. Cadastral only (all polygons)      : {n_cadastral_only:>8,}")

## 6. Gap-Filling

**Case A:** Cadastral geometry exists but height ≤ 0 → fill from nearest GBA building within 2 m buffer.

**Case B:** GBA building has no cadastral counterpart at all → append polygon + height directly.

In [ ]:
buildings_final = cadastral_clipped[["geometry", "height_filtered"]].copy()
buildings_final = buildings_final.rename(columns={"height_filtered": "height"})
buildings_final["source"] = "cadastral"

n_cad_base       = len(buildings_final)
n_cad_has_height = (buildings_final["height"] > 0).sum()
n_needs_fill     = (buildings_final["height"] <= 0).sum()

# Case A: fill missing heights from GBA (2 m buffer)
needs_fill_mask = buildings_final["height"] <= 0
needs_fill      = buildings_final[needs_fill_mask].copy()
needs_fill_buf  = needs_fill.copy()
needs_fill_buf["geometry"] = needs_fill_buf.geometry.buffer(2)

gba_valid   = global_atlas[global_atlas["height_filtered"] > 0][["geometry", "height_filtered"]].copy()
joined_fill = gpd.sjoin(needs_fill_buf[["geometry"]], gba_valid, how="left", predicate="intersects")
best_fill   = joined_fill.dropna(subset=["height_filtered"]).groupby(level=0)["height_filtered"].first()

buildings_final.loc[best_fill.index, "height"] = best_fill.values
buildings_final.loc[best_fill.index, "source"] = "cadastral_gba_fill"

n_filled        = len(best_fill)
n_still_missing = (buildings_final["height"] <= 0).sum()

# Case B: append GBA-only buildings (height > 0 only)
gba_no_cadastral_h = gba_no_cadastral[gba_no_cadastral["height_filtered"] > 0].copy()
gba_extra = gba_no_cadastral_h[["geometry", "height_filtered"]].rename(
    columns={"height_filtered": "height"}
).copy()
gba_extra["source"] = "gba_only"
n_gba_appended = len(gba_extra)

buildings_final = pd.concat([buildings_final, gba_extra], ignore_index=True)
buildings_final = gpd.GeoDataFrame(buildings_final, geometry="geometry", crs=CRS_METRIC)

print("GAP-FILLING")
print(f"  Cadastral base                   : {n_cad_base:>8,}")
print(f"  Already had height               : {n_cad_has_height:>8,}")
print(f"  Needed fill                      : {n_needs_fill:>8,}")
print(f"  Filled from GBA (Case A)         : {n_filled:>8,}")
print(f"  Still missing after fill         : {n_still_missing:>8,}")
print(f"  GBA-only buildings appended (B)  : {n_gba_appended:>8,}")
print(f"  Final total                      : {len(buildings_final):>8,}")

## 7. Save to DuckDB

In [ ]:
# Intermediate layers (useful for QGIS inspection)
gdf_to_duckdb(con, cadastral_clipped, f"preprocessing.{CITY}_buildings_cadastral", CITY)
print(f"  preprocessing.{CITY}_buildings_cadastral: {len(cadastral_clipped):,} features")

gdf_to_duckdb(con, global_atlas, f"preprocessing.{CITY}_buildings_gba", CITY)
print(f"  preprocessing.{CITY}_buildings_gba: {len(global_atlas):,} features")

# Final merged layer (consumed by downstream notebooks)
gdf_to_duckdb(con, buildings_final, f"preprocessing.{CITY}_buildings", CITY)
print(f"  preprocessing.{CITY}_buildings: {len(buildings_final):,} features")
print(buildings_final["source"].value_counts().to_string())

con.close()
print("Done — connection closed.")

## 8. Verify round-trip (optional)

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute("LOAD spatial;")

check = load_table(con, "preprocessing", f"{CITY}_buildings")
print(f"Round-trip check: {len(check):,} buildings")
print(f"CRS: {check.crs}")
print(f"Bounds sample (should be UTM metric): {check.geometry.iloc[0].bounds}")
print(check[["source", "height"]].groupby("source")["height"].describe().round(1).to_string())

con.close()

## 9. Plots

In [ ]:
# Plot 1: Building coverage stacked bar (% of dataset)
fig, ax = plt.subplots(figsize=(7, 5), facecolor="#F7F5F0")
ax.set_facecolor("#F7F5F0")

matched_pcts = [pct_gba_matched,  pct_cad_matched]
only_pcts    = [pct_gba_only,     pct_cad_only]
matched_ns   = [n_gba_matched_cov, n_cadastral_matched_cov]
only_ns      = [n_gba_only,        n_cadastral_only]

ax.bar(categories, matched_pcts, color="#FE9589", width=0.7, zorder=3)
ax.bar(categories, only_pcts, bottom=matched_pcts,
       color=["#8400FF", "#21E5F3"], width=0.7, zorder=3)

for i, (m, u, rm, ru) in enumerate(zip(matched_pcts, only_pcts, matched_ns, only_ns)):
    ax.text(i, m / 2,      f"{m:.1f}%",  ha="center", va="center", color="white", fontsize=9,   fontweight="bold")
    ax.text(i, m / 2 - 4,  f"({rm:,})", ha="center", va="center", color="white", fontsize=6.5, alpha=0.85)
    if u > 2:
        ax.text(i, m + u / 2,     f"{u:.1f}%",  ha="center", va="center", color="white", fontsize=9,   fontweight="bold")
        ax.text(i, m + u / 2 - 4, f"({ru:,})", ha="center", va="center", color="white", fontsize=6.5, alpha=0.85)

ax.legend(handles=[
    mpatches.Patch(color="#FE9589", label="Matched"),
    mpatches.Patch(color="#8400FF", label="GBA only"),
    mpatches.Patch(color="#21E5F3", label="Cadastral only"),
], fontsize=8, frameon=True, facecolor="#F7F5F0", edgecolor="#E8E4DC",
   loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)

ax.set_ylabel("% of dataset", fontsize=9)
ax.set_ylim(0, 108)
ax.set_title("Building Coverage: GBA vs Cadastral", fontweight="bold", fontsize=11)
ax.tick_params(axis="both", labelsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("plot_building_coverage.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 2: Height distributions
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(global_atlas[global_atlas["height_filtered"] > 0]["height_filtered"],
        bins=40, alpha=0.6, color="#8400FF", label="GBA", density=True)
ax.hist(cadastral_clipped[cadastral_clipped["height_filtered"] > 0]["height_filtered"],
        bins=40, alpha=0.6, color="#21E5F3", label="Cadastral", density=True)
ax.set_xlabel("Building Height (m)", fontsize=9)
ax.set_ylabel("Density", fontsize=9)
ax.set_title("Building Height Distribution: GBA vs Cadastral", fontweight="bold", fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("plot_height_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 3: Height scatter (matched buildings, height > 0 in both)
fig, ax = plt.subplots(figsize=(6, 6))
sample = joined_heights.sample(min(3000, len(joined_heights)), random_state=42)

ax.scatter(sample["height_right"], sample["height_left"], alpha=0.2, s=5, color="#2196F3")

max_h = max(sample["height_right"].max(), sample["height_left"].max())
bias  = sample["signed_diff"].mean()

ax.plot([0, max_h], [0, max_h],         color="red",    linewidth=1.5, linestyle="--", label="Perfect agreement (1:1)")
ax.plot([0, max_h], [bias, max_h+bias], color="orange", linewidth=1.5, linestyle="--", label=f"Mean bias ({bias:+.1f} m)")

ax.set_xlabel("Cadastral Height (m)", fontsize=9)
ax.set_ylabel("GBA Height (m)", fontsize=9)
ax.set_title("Height Agreement: GBA vs Cadastral\nMatched buildings (height > 0 in both)",
             fontweight="bold", fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("plot_height_scatter.png", dpi=150, bbox_inches="tight")
plt.show()